## Generate ga data

# TODO

delete le2 and lyapunov notebooks

resample then add noise

fix akulichev (later)

check maximum frequency with higher sampling rate vs downsampling

what happens with no minimum level. test with pcd and simulated data

change wavelet

experiment with entropy threshold for decomposition

look at entropy and energy of nodes with high kurtosis

check matlab unittest again

implement inverse transform?


inconsistent with seed (r = 3.4e-6, freq_threshold = 200e6) seeds 42, 43

changed reconstructed high and low kurtosis to include clusters with low kurtosis

high kurtosis node found in r = 3.4e-6. seems legit though

100 um bubble has long fake peak in psd comparison when fundamental is separated from rest of the nodes

In [ ]:
import sys
if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

import scipy.ndimage as ndi
import plotly.graph_objects as go
import numpy as np

In [ ]:
import sys
sys.path.append("..")

from wavelet_transform.utils import wavelet_features as extract
import wavelet_transform.utils.bubble_data_manager as bdm
from wavelet_transform.utils.preprocessing import preprocess, add_noise
from wavelet_transform.utils.extensions import to_dB, psd
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.machine_learning import fit_k_means
from wavelet_transform.postprocess import plotting as plot
from wavelet_transform.postprocess.plotting import relabel
import wavelet_transform.utils.statistical_features as sf
import wavelet_transform.utils.PCD_data as pcd

### Get experimental data

In [ ]:
frequency = 1.1 * 1e6
pcd_number = 1
voltage = 80  # 80 or 280
data, t_plot = pcd.get_data((voltage, pcd_number))  # [80, 120, 200, 280], [1, 5, 10]

sampling_rate = np.average(np.diff(t_plot))**-1
print(f"Sampling rate: {sampling_rate/1e6:.2f} MHz")

### Bubbly Dynamics Parameters

In [ ]:
use_bd = True  # whether to use the B&D model to generate data

In [ ]:
if use_bd:
    # Set the sonication pulse parameters
    pac = 1e6  # the amplitude of the sine wave in Pa
    frequency = 1.1 * 1e6  # insonation frequency (Hz)
    period = 1 / frequency  # the period of the acoustic wave at the fundamental frequency 
    n_cycles = 100  # no. of cycles of the sine wave
    parameters = {
        "time_delay": 0 * period,  # time delay, t1
        "pulse_width": 10 *n_cycles * period,  # pulse width, W = 1 / Q
        "phase_initial": 0 * period,  # tau
        "phase_shift": 0,  # np.pi / 4,  # phi
        "no_harmonics": 1,   # k
        "Rdot_init": 0,  # initial bubble's wall velocity in m/s
    }

    # Set sampling rate and noise std
    sampling_rate = 100 * frequency  # sampling rate to generate the time vector
    signal2noise = 30

    r_measure = 1e-4  # distance from the centre of bubble which radiated pressure (acoustic emission from bubble) is calculated over time

    Rinit = 3.4e-6  # initial bubble radius in m

### Get bubble dynamics data

In [ ]:
if use_bd:
    data, t_plot = bdm.get_data(
        Rinit, frequency, pac, sampling_rate, n_cycles=n_cycles, r_measure=r_measure,
        return_time=True, **parameters
    )

    # Add noise
    np.random.seed(64)
    data = add_noise(data, signal2noise)
    np.random.seed(None)

In [ ]:
std_original = np.std(data)

### Set wavelet parameters

In [ ]:
packet_bandwidth = frequency / 3
required_levels = int(np.ceil(np.log2(sampling_rate / 2 / packet_bandwidth))); required_levels

In [ ]:
# Parameters for the wavelet transform
wavelet = "db30"
levels = None # set to None to use max-level transform
mode = "symmetric"
use_min_entropy = True
min_level = required_levels
freq_threshold = 1  # 20e6 / (sampling_rate / 2)  # the nodes with norm_freq_range above this threshold will not be kept
fft_lim = 10  # this is the upper limit for plotting the spectrum (fft/psd of emissions)

threshold = 3.2

### Set k-means parameters

In [ ]:
# k-means parameters
kmeans_params = {
    "init": "k-means++",
    "n_init": 20,
    "max_iter": 10,
    "tol": 1e-20,
    "verbose": 0,
    "algorithm": "lloyd",
}

# Training data
classifying_metrics = [
    "log average energy",
    "kurtosis",
    "spectral entropy",
    "average energy",
    # "level entropy",
    # "skewness"
    # "entropy",
    # "crest factor",
]

use_inverse_metrics = True  # Set to None to select optimal number of clusters by silhouette score
keep_ratio = 1  # coefficients kept to calculate statistical metrics
inv_keep_ratio = 1  # keep ratio for inverse defined only for float input

no_clusters = [3, 6]

### Process data

In [ ]:
# Normalise
data_preprocessed = preprocess(data, window=False)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=t_plot, y=data)
)

### DWPT

In [ ]:
wpt_df, tr_obj = extract.extract_wavelet_features(
    data_preprocessed, levels, wavelet, mode,
    include_tr_obj=True, min_entropy=use_min_entropy,
    min_level=min_level, freq_threshold=freq_threshold,
    keep_ratio=keep_ratio
)

In [ ]:
n_nodes = len(tr_obj.get_leaf_nodes(freq_threshold=freq_threshold))
if n_nodes < no_clusters[1]:
    print("not enough nodes. reducing number of kmeans clusters")
no_clusters[1] = min(no_clusters[1], n_nodes)  # set maximum number of clusters 
assert no_clusters[0] <= no_clusters[1], f"too few nodes {n_nodes}"

### Clustering

In [ ]:
if use_inverse_metrics is True:
    inv_tr_obj = inv.InverseTransform(tr_obj)  # Use info from inverse transform object
    wpt_df_train = inv_tr_obj.info(
        leaf_nodes=True, freq_threshold=freq_threshold, keep_ratio=inv_keep_ratio
    )
else:
    wpt_df_train = wpt_df.copy()

data_train, labels, data_train_normalised, centers, scaler, kmeans = fit_k_means(
    wpt_df_train, 
    classifying_metrics,
    no_clusters=tuple(no_clusters),
    kmeans_params=kmeans_params, 
)
centers_scaled = scaler.fit_transform(centers)
centers_sorted, labels_sorted = relabel(labels, centers, "average energy", ascending=False)

In [ ]:
centers_sorted

In [ ]:
fig = plot.plot_clustering(
    data_train, 
    centers_sorted, 
    labels_sorted, 
    [classifying_metrics[i] for i in [0, 1, 2]], 
    title=None
)

In [ ]:
Font_Size = 24
Tick_Size = 24
Fig_Height = 400
Fig_Width = 800

In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
fig = plot.plot_fft_by_group(
    tr_obj, labels_sorted, sampling_rate,
    clusters=None,
    xlims=[0,fft_lim],
    ylims=[-100,50],
    fundamental_freq=frequency,
    freq_threshold=freq_threshold,
    fig_height=Fig_Height,
    fig_width=Fig_Width,
    plot_full_reconstruction=False
)

fig = plot.plot_reconstructions_by_group(
    tr_obj, labels_sorted, t_plot, freq_threshold,
    fig_height=Fig_Height,
    fig_width=Fig_Width,
)

In [ ]:
def trim(data, ratio):
    lim = int((1 - ratio) / 2 * len(data))
    if lim == 0:
        return data
    return data[lim:-lim]

# Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
# plot_centres = [i for i in range(5)]
plot_centres = np.arange(len(centers_sorted)) 
plot_centres = plot_centres[centers_sorted['kurtosis']>threshold]
reconstructions = [np.zeros(len(data_preprocessed)) for _ in range(2)]

for ii, (node, assignment) in enumerate(
    zip(tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels_sorted)
):
    if int(assignment) in plot_centres:
        # List reconstructions
        rec_signal = inv.inv_dwpt(tr_obj, node.level, nodes=node)

        j = 0 if sf.kurtosis(trim(rec_signal, inv_keep_ratio)) > threshold else 1  # shorten name for readability
        reconstructions[j] = reconstructions[j] + rec_signal
    else:
        rec_signal = inv.inv_dwpt(tr_obj, node.level, nodes=node)
        reconstructions[1] += rec_signal


fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=reconstructions[0],
        mode="lines",
        name="higher than threshold",
    )
)
fig.add_trace(
    go.Scatter(
        y=reconstructions[1],
        mode="lines",
        name="lower than threshold",
    )
)


In [ ]:
psd_rec_higher, freq_higher = psd(reconstructions[0], sampling_rate, frequency)
psd_rec_lower, freq_lower = psd(reconstructions[1], sampling_rate, frequency)

# Generate polynomial fitting of the PSD curve
psd_curve_db = ndi.gaussian_filter1d(to_dB(psd_rec_higher), 30)
psd_curve = 10 ** (psd_curve_db / 10)

rms = np.sqrt(np.mean(psd_curve**2))
half_peak = max(psd_curve) / 2
e_peak = max(psd_curve) / np.e**2

broadband_lines = {
    "rms": rms, 
    "FWHM": half_peak, 
    "1/e<sup>2</sup> peak": e_peak
}

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=to_dB(psd_rec_higher),
        mode="lines",
        name="higher than threshold",
    )
)
fig.add_trace(
    go.Scatter(
        x=freq_higher,
        y=psd_curve_db,
        name="envelope"
    )
)

for line_name, line in broadband_lines.items():
    db_line = 10 * np.log10(line)
    fig.add_trace(
        go.Scatter(
            x=freq_higher,
            y=db_line*np.ones(len(freq_higher)),
            line=dict(dash="dash"),
            name=line_name
        )
    )
fig.update_layout(
    xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
)

In [ ]:
print(std_original)

In [ ]:
print("Energy of the HIGH-kurtosis broadband [dB]: ",sf.avg_energy(trim(reconstructions[0], inv_keep_ratio))*std_original, 
      "\nEnergy of the LOW-kurtosis broadband [dB]:",sf.avg_energy(trim(reconstructions[1], inv_keep_ratio))*std_original)

In [ ]:
print("Kurtosis of the HIGH-kurtosis broadband: ",sf.kurtosis(trim(reconstructions[0], inv_keep_ratio)), 
      "\nKurtosis of the LOW-kurtosis broadband:",sf.kurtosis(trim(reconstructions[1], inv_keep_ratio)))

In [ ]:
tr_obj.max_level()

In [ ]:
tr_obj.plot_tree(freq_threshold=freq_threshold, node_size="average energy", node_colour=list(labels_sorted), )

In [ ]:
# Print kurtosis of child node and all parent nodes until level 1
i = 1
for node in tr_obj.get_leaf_nodes(freq_threshold=freq_threshold):
    if node.kurtosis > 7 and i < 5:
        fig = go.Figure()
        i = 1
        print(f"frequency range: {np.asarray(node.numeric_freq_range) * (sampling_rate / 2)/1e6} MHz")
        while node.level > 0:
            rec_signal = inv.inv_dwpt(tr_obj, node.level, nodes=node)
            rec_signal_trimmed = trim(rec_signal, inv_keep_ratio)
            print(f"Level {node.level} - Node {node.node_number} - Kurtosis: {sf.kurtosis(rec_signal_trimmed)} -"
                  f"Energy: {sf.energy(rec_signal_trimmed)}")
            fig.add_trace(
                go.Scatter(y=rec_signal_trimmed, mode='lines', name=f"Level {node.level} - Node {node.node_number}")
            )
            node = node.parent
        fig.show()
        i += 1